In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]  # notebooks/analysis -> project root
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "openslr104"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MIN_WORDS = 3  # same rule as the train/test leak check


def light_norm(s: str) -> str:
    return " ".join(unicodedata.normalize("NFC", s).lower().split())


def read_manifest(split: str) -> pd.DataFrame:
    df = pd.read_json(PROCESSED_DIR / f"{split}_manifest.jsonl", lines=True)
    df["text_key"] = df["text"].map(light_norm)
    df["n_words"] = df["text_key"].str.split().str.len()
    return df


train = read_manifest("train")
train_keys = set(train["text_key"])

for split in ("validation", "test"):
    df = read_manifest(split)
    assert not set(df["recording_id"]) & set(train["recording_id"]), f"a {split} recording is in train"
    leak = df["text_key"].isin(train_keys) & (df["n_words"] >= MIN_WORDS)
    print(
        f"{split}: {leak.sum():,} of {len(df):,} utterances leaked ({leak.mean():.1%}), "
        f"{df.loc[leak, 'n_words'].sum() / df['n_words'].sum():.1%} of words, "
        f"{df.loc[leak, 'duration'].sum() / 3600:.2f} h"
    )
    if split == "validation":
        val, val_leak = df, leak

validation: 185 of 2,058 utterances leaked (9.0%), 7.3% of words, 0.24 h
test: 101 of 2,942 utterances leaked (3.4%), 2.7% of words, 0.13 h


In [2]:
val.loc[val_leak, ["utt_id", "text"]].to_csv(INTERIM_DIR / "openslr104_validation_leaked.csv", index=False)
val.loc[val_leak, "text_key"].value_counts().head(20)

text_key
और enter दबाएं                                                                                                       10
save पर click करें                                                                                                    5
आपका पूर्ण नियत कार्य इस प्रकार दिखना चाहिए                                                                           4
अधिक जानकारी के लिए कृपया हमें लिखें                                                                                  3
dialog box बंद करें                                                                                                   3
इस mission पर अधिक जानकारी निम्न link पर उपलब्ध है spokentutorial org nmeictintro                                     2
यह script प्रभाकर द्वारा अनुवादित है आई आई टी bombay से मैं श्रुति आर्य अब आपसे विदा लेती हूँ                         2
इस tutorial को record करने के लिए मैं उपयोग कर रही हूँ                                                                2
fossee team tbc project का संयो